# 3 · Bonus: a small CNN on a wind field

The wind on a 9 × 9 grid around Bratislava, treated as an image with two channels, u and v. A small convolutional network predicts the wind at the centre 6 hours ahead, trained by the loop of notebook 2. Background is in the [notes](notes/03_cnn.md).

The tools, and the data: the 9 × 9 field for 2023–2024, shipped in `data/`; set `SOURCE = "open-meteo"` to download it again.

In [ ]:
import math
from time import sleep
import numpy as np
import torch
from torch import nn
from windlab import Standardize, fetch_wind, field_samples, fit, least_squares, mlp, patch, predict, rmse, warmup_cosine, year
from plots import hardest_week, plot_convolution, plot_field, plot_forecasts

SOURCE = "local"  # "local": the snapshot in data/ (no network); "open-meteo": download the same data again
lats, lons = patch(48.15, 17.11, radius=4)  # 9 x 9 ERA5 points, 2 x 2 degrees around Bratislava
if SOURCE == "local":
    data = np.load("data/bratislava_field9_2023-2024.npz")  # stored as float16 to keep the repository small
    time, u, v = data["time"], data["u"].astype(np.float32), data["v"].astype(np.float32)
else:  # two calls of about 420 weighted API calls each, a minute apart (the free limit is 600 per minute)
    parts = []
    for i, yr in enumerate(("2023", "2024")):
        sleep(65 * i)
        parts.append(fetch_wind(lats, lons, f"{yr}-01-01", f"{yr}-12-31"))
    time, u, v = (np.concatenate(p) for p in zip(*parts))
GRID, centre = (9, 9), len(lats) // 2

## 1 · The wind field as an image

The code picks the hour with the most structure across the field and draws it: arrows for the wind, coloured by speed, and a red cross on the point we forecast.

In [ ]:
hour = np.argmax(u.std(axis=1) + v.std(axis=1))  # the hour with the most structure across the field
plot_field(lons, lats, u, v, hour, GRID, centre, time)

## 2 · Samples: a stack of fields

**Input** $x$: the field at hours $t-2$, $t-1$ and $t$, a tensor of 6 channels × 9 × 9. **Target** $y$: u and v at the centre at hour $t + 6$, learned in the delta form $\hat y = \text{wind now} + f(x;\theta)$. Train on 2023, test on 2024.

The code builds these samples for every hour and reshapes each into 6 channels × 9 × 9.

In [ ]:
LAGS, HORIZON = 3, 6
X, Y, now, t = field_samples(u, v, LAGS, HORIZON)  # inputs x, targets y, the wind now (= persistence), the hour t
X = X.reshape(len(X), 2 * LAGS, *GRID)  # (samples, channels, latitude, longitude)
train, test = year(time[t]) == 2023, year(time[t]) == 2024

## 3 · A convolutional block

A 3 × 3 convolution slides a small stencil of weights $w$ over the map: at every grid point it multiplies the point and its 8 neighbours by the weights and adds them up. The code draws this for one stencil, "wind in the west minus wind in the east", on u at the hour of section 1: the input with one window, the stencil, and the output at every point.

In [ ]:
stencil = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]]) / 3  # rows: south to north; columns: west to east
plot_convolution(u[hour].reshape(GRID), stencil, point=(4, 4))

**Look for:** the output is positive (red) where u falls from west to east, so that air piles up, and negative (blue) where it spreads out. The same stencil is used at every point, so a pattern is recognised wherever it appears. A network is not given its stencils: one convolution learns 16 of them, each across all 6 input channels, plus a bias $b$ for each,

$$z_{c',i,j} = b_{c'} + \sum_{c}\sum_{p=-1}^{1}\sum_{q=-1}^{1} w_{c',c,p,q}\; x_{c,\,i+p,\,j+q}.$$

Animated pictures of convolutions: [conv_arithmetic](https://github.com/vdumoulin/conv_arithmetic).

The code builds the block (two such convolutions with GELU), the CNN (the block, then a linear layer for the two outputs) and, for comparison, the MLP of notebook 2, which gets the same field as a flat list of 486 numbers. Both are trained in the delta form, 20 epochs each.

In [ ]:
class ConvBlock(nn.Module):
    # Two 3 x 3 convolutions with GELU; padding=1 keeps the grid size
    def __init__(self, c_in, c_out):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(c_in, c_out, kernel_size=3, padding=1), nn.GELU(),
            nn.Conv2d(c_out, c_out, kernel_size=3, padding=1), nn.GELU(),
        )

    def forward(self, x):
        return self.layers(x)

def cnn(x_train, channels=16):
    c_in, height, width = x_train.shape[1:]
    return nn.Sequential(Standardize(x_train), ConvBlock(c_in, channels), nn.Flatten(), nn.Linear(channels * height * width, 2))

def dense(x_train):
    # The MLP of notebook 2 on the flattened field
    return nn.Sequential(nn.Flatten(), mlp(x_train.reshape(len(x_train), -1), n_out=2))

def train_delta(make_model, epochs=20, batch_size=256, seed=0):
    # The training loop of notebook 2 (windlab.fit), on the change from now
    torch.manual_seed(seed)
    model = make_model(X[train])
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-2)
    fit(model, X[train], Y[train] - now[train], opt, epochs=epochs, batch_size=batch_size,
        scheduler=warmup_cosine(opt, epochs * math.ceil(train.sum() / batch_size)), seed=seed)
    return model

mlp_model, cnn_model = train_delta(dense), train_delta(cnn)
for name, model in [("MLP", mlp_model), ("CNN", cnn_model)]:
    print(f"{name}: {sum(p.numel() for p in model.parameters())} parameters")

## 4 · Forecasts against the truth

The code scores five forecasts on 2024: persistence, least squares on the centre point alone and on the whole field, and the two networks.

In [ ]:
def linear(inputs):
    # Least squares as in notebook 2: fitted on 2023, forecast for 2024
    W, b = least_squares(inputs[train], Y[train])
    return inputs[test] @ W + b

point, _, _, _ = field_samples(u[:, [centre]], v[:, [centre]], LAGS, HORIZON)  # the same hours, centre point only
forecasts = {
    "persistence": now[test],
    "linear, centre point": linear(point),
    "linear, whole field": linear(X.reshape(len(X), -1)),
    "MLP, whole field": now[test] + predict(mlp_model, X[test]),
    "CNN, whole field": now[test] + predict(cnn_model, X[test]),
}
for name, forecast in forecasts.items():
    print(f"{name:22s} RMSE {rmse(forecast, Y[test]):.3f} m/s, skill {1 - rmse(forecast, Y[test]) / rmse(now[test], Y[test]):+.3f}")

The week of 2024 in which persistence misses most, chosen as in notebooks 1 and 2: the truth (black), persistence, the centre-point line and the CNN.

In [ ]:
week = hardest_week(now[test, 0] - Y[test, 0])  # positions within the test samples
plot_forecasts(time[t[test][week] + HORIZON], Y[test][week, 0],
               {name: forecasts[name][week, 0] for name in ("persistence", "linear, centre point", "CNN, whole field")},
               ylabel=f"y: u at the centre in {HORIZON} h (m/s)", title="2024: the week in which persistence does worst")

**Look for:** the whole field more than doubles the skill of the centre point alone, because wind upstream arrives later; the CNN gets the most out of it with the fewest parameters, but is still late for the sharpest turns.

## Your turn (after the workshop)

1. Set `channels` to 8 or 32, or stack a second `ConvBlock`. Do more parameters help?
2. Try `HORIZON = 12` or `LAGS = 1`. When does the whole field help most?
3. In the drawing of section 3, use `stencil.T` on `v[hour]`: the same idea from south to north.